# **DIDP model and dual bound declaration**

In [1]:
%%writefile v1_CVRP_DIDP_model_and_dual_bound_declaration.py
import sys
import os
import re
import numpy as np
import vrplib # <--- Added for your reader
import modified_didppy as m_dp
from ortools.linear_solver import pywraplp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
from functools import lru_cache
from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry

# ==========================================
# GLOBAL VARIABLES (Your Naming Convention)
# ==========================================
# --- Problem Setup ---
current_capacity = 1000              # Capacity set to 1000
current_num_locations = 2            # Two locations
current_num_vehicles = 0             # Still zero vehicles (adjust if needed)

# Customer demands: two entries, both 0
current_cust_demands = [0, 0]

# Travel cost: 2x2 matrix with all entries = 0
current_travel_cost = [
    [0, 0],
    [0, 0]
]

# ==========================================
# 1. YOUR READER FUNCTION
# ==========================================
def update_globals_for_cvrp(file_path):
    """Updates the specific global variables required by creation_of_didp_model_function."""
    global current_capacity, current_num_locations, current_num_vehicles, current_cust_demands, current_travel_cost, current_optimal_cost
    
    instance = vrplib.read_instance(file_path)
    
    # 1. Update Capacity & Dimensions
    current_capacity = instance['capacity']
    current_num_locations = instance['dimension']
    
    # 2. Update Num Vehicles (Regex or Fallback)
    match_trucks = re.search(r"No of trucks:\s*(\d+)", instance.get('comment', ''))
    if match_trucks:
        current_num_vehicles = int(match_trucks.group(1))
    else:
        # Fallback logic for X-series or if comment is missing
        match_filename = re.search(r"-k(\d+)", os.path.basename(file_path))
        if match_filename:
            current_num_vehicles = int(match_filename.group(1))
        else:
            current_num_vehicles = 25 # Fallback
            
    # 3. Update Demands & Costs
    current_cust_demands = instance['demand']
    current_travel_cost = instance['edge_weight']
    
    return True

def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    # =========================================================
    # 1. Define Data
    # =========================================================
    n = current_num_locations
    m = current_num_vehicles
    q = current_capacity
    # Weights (demand)
    d = current_cust_demands

    # Distance matrix
    distance_list = current_travel_cost
    
    # =========================================================
    # 2. Define DIDP model
    # =========================================================
    model = m_dp.Model(float_cost= True)

    customer = model.add_object_type(number=n)
    unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name='unvisited_customers')
    location_var = model.add_element_var(object_type=customer, target=0)
    load_var = model.add_float_resource_var(target=0, less_is_better=True)
    vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)

    weight = model.add_float_table(d)
    distance_table = model.add_float_table(distance_list)

    model.add_base_case([unvisited_var.is_empty(), location_var == 0])

    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}",
            cost=distance_table[location_var, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, load_var + weight[j]),
            ],
            preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
        )
        model.add_transition(visit)

    for j in range(1, n):
        visit_via_depot = m_dp.Transition(
            name=f"visit {j} with new vehicle",
            cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, weight[j]),
                (vehicles_var, vehicles_var + 1),
            ],
            preconditions=[unvisited_var.contains(j), vehicles_var < m],
        )
        model.add_transition(visit_via_depot)

    return_to_depot = m_dp.Transition(
        name="return",
        cost=distance_table[location_var, 0] + m_dp.FloatExpr.state_cost(),
        effects=[(location_var, 0)],
        preconditions=[unvisited_var.is_empty(), location_var != 0],
    )
    model.add_transition(return_to_depot)

    model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])

    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unvisited_var": unvisited_var,
        "location_var": location_var,
        "distance_matrix": distance_list,
        "demand": d,
        "capacity": q,
        "num_vehicles": m,
        "num_nodes": n
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    """
    Creates a persistent 3-Index CVRP Relaxation (Vehicle-Node-Node).
    - Logic: VRP4 Model (Assignment + Flow + MTZ Capacity).
    - Pattern: Cached internal worker '_solve_3idx'.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    n_vehicles = metadata['num_vehicles']
    capacity = metadata['capacity']
    demands = metadata['demand']
    dist_matrix = metadata['distance_matrix']
    
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # --- INITIALIZATION (Runs Once) ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    infinity = solver.infinity()

    # --- Variables ---
    x = {} # Flow x_k_i_j
    y = {} # Assignment y_i_k
    u = {} # Load u_i_k

    # 1. Flow
    for k in range(n_vehicles):
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j: x[(k, i, j)] = solver.NumVar(0, 1, f'x_{k}_{i}_{j}')

    # 2. Assignment (All nodes, All vehicles)
    for i in range(n_nodes):
        for k in range(n_vehicles):
            y[(i, k)] = solver.NumVar(0, 1, f'y_{i}_{k}')

    # 3. Potentials (Load)
    for i in range(1, n_nodes):
        for k in range(n_vehicles):
            u[(i, k)] = solver.NumVar(0, capacity, f'u_{i}_{k}')

    # --- Constraints ---
    
    # (1.29) Customer Assignment (Mutable)
    # This is what we toggle on/off based on the state.
    cons_assignment = {}
    for i in range(1, n_nodes):
        c = solver.Constraint(0, 0, f'assign_{i}')
        for k in range(n_vehicles): c.SetCoefficient(y[(i, k)], 1)
        cons_assignment[i] = c

    # (1.30) Depot Usage
    c_depot = solver.Constraint(0, n_vehicles, 'depot_usage')
    for k in range(n_vehicles): c_depot.SetCoefficient(y[(0, k)], 1)

    # (1.31) Flow Conservation
    for k in range(n_vehicles):
        for i in range(n_nodes):
            # Out
            c_out = solver.Constraint(0, 0, f'flow_out_{i}_{k}')
            c_out.SetCoefficient(y[(i, k)], -1)
            for j in range(n_nodes):
                if i != j: c_out.SetCoefficient(x[(k, i, j)], 1)
            
            # In
            c_in = solver.Constraint(0, 0, f'flow_in_{i}_{k}')
            c_in.SetCoefficient(y[(i, k)], -1)
            for j in range(n_nodes):
                if i != j: c_in.SetCoefficient(x[(k, j, i)], 1)

    # (1.37) & (1.38) MTZ Capacity
    for k in range(n_vehicles):
        for i in range(1, n_nodes):
            for j in range(1, n_nodes):
                if i != j:
                    c = solver.Constraint(-infinity, float(capacity - demands[j]), f'mtz_{k}_{i}_{j}')
                    c.SetCoefficient(u[(i, k)], 1)
                    c.SetCoefficient(u[(j, k)], -1)
                    c.SetCoefficient(x[(k, i, j)], capacity)

    # --- Objective ---
    objective = solver.Objective()
    for k in range(n_vehicles):
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j: objective.SetCoefficient(x[(k, i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # ==========================================
    # 2. CACHED SOLVER WORKER
    # ==========================================
    @lru_cache(maxsize=500000)
    def _solve_3idx(active_tuple):
        """
        Internal worker: Sets bounds and solves.
        active_tuple: Tuple of customer indices that MUST be visited.
        """
        active_set = set(active_tuple)

        # Reset/Update Assignment Constraints
        for i in range(1, n_nodes):
            if i in active_set:
                # ACTIVE: Must be visited
                cons_assignment[i].SetBounds(1, 1)
                # Load variables active
                for k in range(n_vehicles):
                    u[(i, k)].SetBounds(float(demands[i]), float(capacity))
            else:
                # INACTIVE: Must NOT be visited (flow = 0)
                cons_assignment[i].SetBounds(0, 0)
                # Load variables forced to 0
                for k in range(n_vehicles):
                    u[(i, k)].SetBounds(0, 0)

        # Solve
        solver.SetTimeLimit(100) # 100ms
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # ==========================================
    # 3. HEURISTIC WRAPPER
    # ==========================================
    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_var]
        current_loc = state[location_var]
        
        # Optimization: Solved state
        if not unvisited and current_loc == 0: 
            return 0.0

        # Define Active Set: Unvisited + Current (if not depot)
        active_customers = set(unvisited)
        if current_loc != 0:
            active_customers.add(current_loc)
        
        # Create Tuple Key
        active_key = tuple(sorted(list(active_customers)))
        
        return _solve_3idx(active_key)

    return h_lp_relaxation_3_idx

def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    """
    Creates a persistent 2-Index CVRP Relaxation (Flow-based).
    - Logic: Aggregated Flow (No Vehicle Dimension).
    - Pattern: Cached internal worker '_solve_2idx'.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    n_vehicles = metadata['num_vehicles']
    capacity = metadata['capacity']
    demands = metadata['demand']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']

    # --- INITIALIZATION ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    infinity = solver.infinity()

    # --- Variables ---
    x = {} # Flow x_i_j
    u = {} # Load u_i

    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    for i in range(1, n_nodes):
        u[i] = solver.NumVar(0, capacity, f'u_{i}')

    # --- Constraints ---
    cons_degree_out = {}
    cons_degree_in = {}

    # 1. Degree Constraints (Mutable)
    for i in range(1, n_nodes):
        # Outgoing
        c_out = solver.Constraint(0, 0, f'deg_out_{i}')
        for j in range(n_nodes):
            if i != j: c_out.SetCoefficient(x[(i, j)], 1)
        cons_degree_out[i] = c_out

        # Incoming
        c_in = solver.Constraint(0, 0, f'deg_in_{i}')
        for j in range(n_nodes):
            if i != j: c_in.SetCoefficient(x[(j, i)], 1)
        cons_degree_in[i] = c_in

    # 2. Depot Degree
    c_depot_out = solver.Constraint(0, n_vehicles, 'depot_out')
    c_depot_in = solver.Constraint(0, n_vehicles, 'depot_in')
    for j in range(1, n_nodes):
        c_depot_out.SetCoefficient(x[(0, j)], 1)
        c_depot_in.SetCoefficient(x[(j, 0)], 1)

    # 3. MTZ Capacity
    for i in range(1, n_nodes):
        for j in range(1, n_nodes):
            if i != j:
                c = solver.Constraint(-infinity, float(capacity - demands[j]), f'mtz_{i}_{j}')
                c.SetCoefficient(u[j], 1)
                c.SetCoefficient(u[i], -1)
                c.SetCoefficient(x[(i, j)], capacity)

    # --- Objective ---
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # ==========================================
    # 2. CACHED SOLVER WORKER
    # ==========================================
    @lru_cache(maxsize=500000)
    def _solve_2idx(active_tuple):
        active_set = set(active_tuple)

        # Reset/Update Constraints
        for i in range(1, n_nodes):
            if i in active_set:
                # ACTIVE
                cons_degree_out[i].SetBounds(1, 1)
                cons_degree_in[i].SetBounds(1, 1)
                u[i].SetBounds(float(demands[i]), float(capacity))
            else:
                # INACTIVE
                cons_degree_out[i].SetBounds(0, 0)
                cons_degree_in[i].SetBounds(0, 0)
                u[i].SetBounds(0, 0)
        
        solver.SetTimeLimit(100)
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # ==========================================
    # 3. HEURISTIC WRAPPER
    # ==========================================
    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        
        if not unvisited: return 0.0

        # Define Active Set: Unvisited
        # Note: 2-index relaxation often ignores "current location" specific routing logic 
        # and just solves the flow problem for the remaining set.
        active_key = tuple(sorted(list(unvisited)))

        return _solve_2idx(active_key)

    return h_lp_relaxation_2_idx

def dual_bound_expression_function(didp_bundle):
    """ 
    Registry containing ALL heuristics (Combinatorial + LP) for CVRP.
    """
    model, metadata = didp_bundle
    
    # Extract metadata
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list)
    demand = metadata['demand']
    capacity = metadata['capacity']
    num_vehicles = metadata['num_vehicles']
    n_nodes = metadata['num_nodes']

    # Pre-computation
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    min_outgoing_arr = np.min(masked_cost, axis=1)
    min_incoming_arr = np.min(masked_cost, axis=0)

    # --- Initialize LP Bounds ---
    h_lp_3idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata)
    h_lp_2idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata)

    # ==========================================
    # COMBINATORIAL BOUNDS (Internal Workers)
    # ==========================================

    # --- Flow Bound ---
    @lru_cache(maxsize=500000)
    def _calc_flow(unvisited_tuple):
        s = sum(demand[v] * distance_list[0][v] for v in unvisited_tuple)
        return float(round((2.0 / capacity) * s))

    @lru_cache(maxsize=10000)
    def h_flow(state):
        U = state[unvisited_var]
        return _calc_flow(tuple(sorted(list(U)))) if U else 0.0

    # --- Degree Average Bound ---
    @lru_cache(maxsize=500000)
    def _calc_degree(active_tuple):
        nodes = list(active_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)
        sum_in = np.sum(np.min(sub_mat, axis=0))
        sum_out = np.sum(np.min(sub_mat, axis=1))
        return float(0.5 * (sum_in + sum_out))

    @lru_cache(maxsize=10000)
    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        # Active: Current -> Unvisited -> Depot
        active = set(U)
        active.add(curr)
        active.add(0)
        return _calc_degree(tuple(sorted(list(active))))

    # --- Global Min Flow ---
    @lru_cache(maxsize=500000)
    def _calc_min_flow_static(unvisited_tuple):
        val_out = sum(min_outgoing_arr[u] for u in unvisited_tuple)
        val_in = sum(min_incoming_arr[u] for u in unvisited_tuple)
        return val_out, val_in

    @lru_cache(maxsize=10000)
    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        # Get static part
        val_out, val_in = _calc_min_flow_static(tuple(sorted(list(U))))
        
        # Add dynamic part (Current)
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            val_in += min_incoming_arr[0]
            
        return float(max(val_out, val_in))

    # --- MST Bound ---
    @lru_cache(maxsize=500000)
    def _calc_mst(unvisited_tuple):
        if not unvisited_tuple: return 0.0
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    @lru_cache(maxsize=10000)
    def h_mst(state):
        U = state[unvisited_var]
        return _calc_mst(tuple(sorted(list(U))))

    # --- 1-Tree Bound ---
    @lru_cache(maxsize=500000)
    def _calc_1tree(unvisited_tuple):
        subset = list(unvisited_tuple)
        depot_edges = sorted(cost_matrix[0, subset])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        
        if len(subset) > 1:
            sub_mat = cost_matrix[np.ix_(subset, subset)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0
        return float(mst_val + e1 + e2)

    @lru_cache(maxsize=10000)
    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_1tree(tuple(sorted(list(U))))

    # --- Assignment Bound ---
    @lru_cache(maxsize=500000)
    def _calc_assignment(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        row, col = linear_sum_assignment(assign_mat)
        return float(assign_mat[row, col].sum())

    @lru_cache(maxsize=10000)
    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_assignment(tuple(sorted(list(U))))

    # --- Eigenvalue Bound ---
    @lru_cache(maxsize=500000)
    def _calc_eigen(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        N = len(nodes)
        if N < 2: return 0.0
        
        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])
        
        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals): phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
        return float(phi)

    @lru_cache(maxsize=10000)
    def h_eigen(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_eigen(tuple(sorted(list(U))))

    # Return Registry
    return automatic_creation_of_dual_bounds_registry(locals())


Overwriting v1_CVRP_DIDP_model_and_dual_bound_declaration.py


# **Worker**

In [2]:
%%writefile worker_cvrp_profiling.py
import sys
import os
import ast
import json
import time
import threading

# --- CONFIGURATION ---
MEMORY_LIMIT_MB = 12000 # Increased to 12GB just in case

# --- 1. SETUP PATHS ---
PROJECT_ROOT = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP"
LIB_PATH = os.path.join(PROJECT_ROOT, "Evolutionary_algorithm")

sys.path.append(os.getcwd()) 
if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)
if LIB_PATH not in sys.path: sys.path.append(LIB_PATH)

# --- 2. IMPORTS ---
try:
    import psutil
    import modified_didppy as m_dp
    import v1_CVRP_DIDP_model_and_dual_bound_declaration as domain 
    from evolutionary_algorithm_lib import compile_chromosome_to_useable_function
except ImportError as e:
    print(json.dumps({"status": "error", "message": f"Import Error: {e}"}))
    sys.exit(1)

def memory_guard():
    process = psutil.Process(os.getpid())
    limit_bytes = MEMORY_LIMIT_MB * 1024 * 1024
    while True:
        try:
            if process.memory_info().rss > limit_bytes:
                os._exit(1)
            time.sleep(1)
        except: break

if __name__ == "__main__":
    try:
        threading.Thread(target=memory_guard, daemon=True).start()

        if len(sys.argv) < 6: 
            raise ValueError("Args mismatch")
        
        instance_name = sys.argv[1]
        chromosome_str = sys.argv[2]
        data_dir = sys.argv[3]
        time_limit = float(sys.argv[4])
        node_limit = int(sys.argv[5])

        # 1. Load Data
        file_path = os.path.join(data_dir, instance_name)
        if not os.path.exists(file_path):
             file_path_vrp = file_path + ".vrp"
             if os.path.exists(file_path_vrp): file_path = file_path_vrp
        
        if not domain.update_globals_for_cvrp(file_path):
            raise ValueError(f"Failed to load instance: {file_path}")

        # 2. DEBUG: Print Problem Stats to Stderr (Manager will capture this)
        debug_info = (
            f"DEBUG: Loaded {instance_name} | "
            f"Nodes: {domain.current_num_locations}, "
            f"Capacity: {domain.current_capacity}, "
            f"Vehicles: {domain.current_num_vehicles}, "
            f"Total Demand: {sum(domain.current_cust_demands)}"
        )
        sys.stderr.write(debug_info + "\n")

        # 3. Create Model
        model, metadata = domain.creation_of_didp_model_function()
        
        # 4. Prepare Dual Bound
        chromosome = ast.literal_eval(chromosome_str)
        dual_bound_funcs = domain.dual_bound_expression_function((model, metadata))
        h_func = compile_chromosome_to_useable_function(
            {'chromosome': chromosome, 'fitness': 0},
            dual_bound_functions_dict=dual_bound_funcs,
            print_code=False
        )
        
        # 5. Create Solver (Set quiet=False to see solver logs in debug mode)
        solver = m_dp.CustomDualBoundCABSv1(
            model=model,
            dual_bound_func=h_func,
            time_limit=time_limit,
            initial_beam_size = 1,
            max_beam_size=1024,
            quiet=False, # Changed to False to see infeasibility message
            print_timing_stats=False
        )
        
        history = [] 
        start_time = time.time()
        
        while True:
            if time.time() - start_time > time_limit: break
            
            solution, is_terminated = solver.search_next()
            
            if solution.cost is not None:
                history.append((solution.expanded, solution.cost))
            
            if is_terminated: break
            if solution.expanded >= node_limit: break

        output = {
            "status": "success",
            "history": history,
            "duration": time.time() - start_time
        }
        print(json.dumps(output))

    except Exception as e:
        print(json.dumps({"status": "error", "message": str(e)}))
        sys.exit(1)

Overwriting worker_cvrp_profiling.py


# **Manager**

In [ ]:
import subprocess
import pandas as pd
import numpy as np
import os
import re
import sys
import json
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================
NODE_EXPANSION_LIMIT = 100_000 
SOLVER_TIME_LIMIT = 1800 

PATH_SET_A = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\A"
PATH_SET_X = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\X"

EA_RESULTS_CSV = "result_CVRP_EA_dual_bound.csv"
REF_RESULTS_CSV = "CVRP_single_dual_bound_selected_results_10s_lim.csv"
OUTPUT_FOLDER = "Test_CVRP_EA_Profiling_Results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def find_cvrp_file(instance_name):
    path_a = os.path.join(PATH_SET_A, instance_name)
    if os.path.exists(path_a): return path_a
    if os.path.exists(path_a + ".vrp"): return path_a + ".vrp"
    
    path_x = os.path.join(PATH_SET_X, instance_name)
    if os.path.exists(path_x): return path_x
    if os.path.exists(path_x + ".vrp"): return path_x + ".vrp"
    return None

def calculate_dense_gaps(history, optimal_cost, limit):
    sample_rate = 100 
    max_steps = limit // sample_rate
    dense_gaps = np.full(max_steps, np.nan)

    if not history: return dense_gaps, sample_rate 

    history.sort(key=lambda x: x[0])
    current_cost = float('inf')
    hist_idx = 0
    
    for i in range(max_steps):
        current_node_count = (i + 1) * sample_rate
        if hist_idx >= len(history) and history[-1][0] < current_node_count:
             break
        while hist_idx < len(history) and history[hist_idx][0] <= current_node_count:
            if history[hist_idx][1] < current_cost:
                current_cost = history[hist_idx][1]
            hist_idx += 1
        if current_cost == float('inf'):
            gap = 100.0 
        else:
            denom = optimal_cost if optimal_cost > 0 else 1.0
            gap = (current_cost - optimal_cost) / denom * 100.0
            if gap < 0: gap = 0.0 
        dense_gaps[i] = gap
    return dense_gaps, sample_rate

def run_cvrp_profiling():
    print(f"🔹 Starting CVRP Profiling (Max Nodes: {NODE_EXPANSION_LIMIT})...")
    
    if not os.path.exists(EA_RESULTS_CSV) or not os.path.exists(REF_RESULTS_CSV):
        print("❌ Input CSV files missing.")
        return
        
    df_ea = pd.read_csv(EA_RESULTS_CSV)
    df_ref = pd.read_csv(REF_RESULTS_CSV)
    df_merged = pd.merge(df_ea, df_ref[['Instance', 'Best Known Cost']], on='Instance', how='inner')
    
    all_dense_gaps = [] 
    
    for i, row in df_merged.iterrows():
        instance = row['Instance']
        chrom = row['Best_Chromosome']
        optimal_cost = row['Best Known Cost']
        safe_name = instance.replace(".vrp", "")
        
        output_csv_path = os.path.join(OUTPUT_FOLDER, f"gap_profile_{safe_name}.csv")
        output_txt_path = os.path.join(OUTPUT_FOLDER, f"solver_log_{safe_name}.txt") # <--- Log File Path
        
        # RESUME CHECK
        if os.path.exists(output_csv_path):
            print(f"[{i+1}/{len(df_merged)}] Skipping {instance} (Already processed).")
            try:
                df_existing = pd.read_csv(output_csv_path)
                if 'Gap_Percent' in df_existing.columns:
                    dense_gaps = df_existing['Gap_Percent'].values
                    if len(dense_gaps) == (NODE_EXPANSION_LIMIT // 100) and not np.all(np.isnan(dense_gaps)):
                        all_dense_gaps.append(dense_gaps)
            except: pass
            continue

        target_path = find_cvrp_file(instance)
        if target_path is None:
            print(f"\n❌ Could not find file for instance: {instance}")
            continue
            
        target_dir = os.path.dirname(target_path)
        target_filename = os.path.basename(target_path)
        
        print(f"\n[{i+1}/{len(df_merged)}] Profiling {instance} (Opt: {optimal_cost})...")
        
        cmd = [
            sys.executable, "-u", "worker_cvrp_profiling.py",
            target_filename, chrom, target_dir, str(SOLVER_TIME_LIMIT), str(NODE_EXPANSION_LIMIT)
        ]
        
        try:
            # Capture output
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            # --- 1. SAVE TXT LOG ---
            with open(output_txt_path, "w", encoding="utf-8") as f:
                f.write(result.stdout)
                if result.stderr:
                    f.write("\n\n=== STDERR ===\n")
                    f.write(result.stderr)
            # -----------------------

            # Print Stats
            debug_match = re.search(r"DEBUG: Loaded .*? \| Nodes: (\d+), Capacity: (\d+), Vehicles: (\d+)", result.stderr)
            if debug_match:
                nodes, cap, veh = debug_match.groups()
                print(f"   ℹ️  Stats: Customers={nodes}, Vehicles={veh}, Capacity={cap}")

            if result.returncode != 0:
                print(f"   ⚠️ Worker Failed (Code {result.returncode})")
                print(f"   🔎 STDERR: {result.stderr}")
                continue
                
            # --- 2. ROBUST JSON PARSING ---
            # Output now contains mixed text. We look for the JSON line.
            json_str = ""
            for line in result.stdout.split('\n'):
                line = line.strip()
                if line.startswith('{') and line.endswith('}'):
                    # Check if it looks like our output object
                    if '"status":' in line:
                        json_str = line
            
            if not json_str:
                print("   ⚠️ No JSON found in output.")
                # print(f"   🔎 RAW OUTPUT:\n{result.stdout[:500]}...") # Optional debug
                continue
                
            data = json.loads(json_str)
            if data['status'] != 'success':
                print(f"   ⚠️ Worker Error: {data.get('message')}")
                continue
                
            history = data['history']
            print(f"   -> Retrieved {len(history)} points. Log saved.")

            if len(history) == 0:
                print(f"   ⚠️ ZERO HISTORY POINTS. Check {output_txt_path}")
                continue 
            
            dense_gaps, step_size = calculate_dense_gaps(history, optimal_cost, NODE_EXPANSION_LIMIT)
            all_dense_gaps.append(dense_gaps)
            
            df_hist = pd.DataFrame({
                'Node_Count': np.arange(step_size, NODE_EXPANSION_LIMIT + 1, step_size),
                'Gap_Percent': dense_gaps
            })
            df_hist.to_csv(output_csv_path, index=False)
            
        except Exception as e:
            print(f"   ❌ Error: {e}")

    # Aggregation
    if all_dense_gaps:
        print("\n📊 Aggregating Results...")
        gap_matrix = np.vstack(all_dense_gaps)
        with np.errstate(invalid='ignore'):
            mean_gaps = np.nanmean(gap_matrix, axis=0)
        
        step_size = 100
        x_axis = np.arange(step_size, NODE_EXPANSION_LIMIT + 1, step_size)
        
        df_agg = pd.DataFrame({
            'Node_Count': x_axis,
            'Average_Gap_Percent': mean_gaps
        })
        
        agg_path = os.path.join(OUTPUT_FOLDER, "CVRP_Average_Gaps_vs_Expansion.csv")
        df_agg.to_csv(agg_path, index=False)
        print(f"   ✅ Saved aggregated data to {agg_path}")
        
        plt.figure(figsize=(10, 6))
        plt.plot(x_axis, mean_gaps, label='Average Gap (%)', color='blue')
        plt.xlabel('Nodes Expanded')
        plt.ylabel('Average Optimality Gap (%)')
        plt.title('CVRP Convergence Profile')
        plt.grid(True)
        plt.legend()
        plt.savefig(os.path.join(OUTPUT_FOLDER, "convergence_plot.png"))
        print(f"   ✅ Saved plot.")
    else:
        print("\n⚠️ No data collected.")

if __name__ == "__main__":
    run_cvrp_profiling()

🔹 Starting CVRP Profiling (Max Nodes: 100000)...
[1/10] Skipping A-n39-k6.vrp (Already processed).
[2/10] Skipping A-n48-k7.vrp (Already processed).
[3/10] Skipping A-n55-k9.vrp (Already processed).
[4/10] Skipping A-n69-k9.vrp (Already processed).
[5/10] Skipping A-n80-k10.vrp (Already processed).
[6/10] Skipping X-n106-k14.vrp (Already processed).
[7/10] Skipping X-n162-k11.vrp (Already processed).

[8/10] Profiling X-n181-k23.vrp (Opt: 25569)...
   ℹ️  Stats: Customers=181, Vehicles=23, Capacity=8
   -> Retrieved 0 points. Log saved.
   ⚠️ ZERO HISTORY POINTS. Check Test_CVRP_EA_Profiling_Results\solver_log_X-n181-k23.txt
[9/10] Skipping X-n190-k8.vrp (Already processed).

[10/10] Profiling X-n204-k19.vrp (Opt: 19565)...
